# Robot Donald Trump

Create new Donald Trump Tweets

In [10]:
import subprocess as sp
import kaggle.api
import pandas as pd
import numpy as np
import sentencepiece as spm
import torch
from pathlib import Path
from codecs import encode, decode
from sys import stdout
from random import choice
from tqdm.notebook import tqdm

# Model params
vocab_size = 1000
latent_size = 256
attention_layers = 2
attention_heads = 2

# Training params
batch_size = 24
num_epochs = 1

# Get accelerator
device = "cpu"
if torch.accelerator.is_available():
    accelerator = torch.accelerator.current_accelerator()
    print('We got an accelerator!:', accelerator)
    device = accelerator.type
print(f"Using device: '{device}'")

# Folders
data_folder = Path('./data/remote/robot-trump')
model_folder = Path('./models/robot-trump')
if not data_folder.exists(): data_folder.mkdir()
if not model_folder.exists(): model_folder.mkdir()

Using device: 'cpu'


Download and pull dataset.

In [11]:
kaggle.api.dataset_download_file(dataset='austinreese/trump-tweets', 
                                 file_name='realdonaldtrump.csv', 
                                 path=str(data_folder))
result = sp.run(['tar', '-xf', 'realdonaldtrump.csv.zip'], cwd=str(data_folder))
print('tar -xf returned', result.returncode)
df = pd.read_csv(str(data_folder / 'realdonaldtrump.csv'))

Dataset URL: https://www.kaggle.com/datasets/austinreese/trump-tweets
tar -xf returned 0


Just going to be using all the text for now. Separate cell in case I want to do filtering later

In [12]:
tweets = df.content.to_numpy().astype(np.dtypes.StringDType())
tweets

array(['Be sure to tune in and watch Donald Trump on Late Night with David Letterman as he presents the Top Ten List tonight!',
       'Donald Trump will be appearing on The View tomorrow morning to discuss Celebrity Apprentice and his new book Think Like A Champion!',
       'Donald Trump reads Top Ten Financial Tips on Late Show with David Letterman: http://tinyurl.com/ooafwn - Very funny!',
       ..., 'pic.twitter.com/3lm1spbU8X', 'pic.twitter.com/vpCE5MadUz',
       'pic.twitter.com/VLlc0BHW41'], shape=(43352,), dtype=StringDType())

Train sentencepiece tokenizer

In [13]:
# Write tweets to file
tweets_file = data_folder / 'tweets.txt'
tweets_file.write_text('\n'.join(tweets), encoding='utf-8')
sentencepiece_prefix = model_folder / 'sentencepiece'
spm.SentencePieceTrainer.train(input=str(tweets_file), model_prefix=str(sentencepiece_prefix), vocab_size=vocab_size)
print('All done!')

All done!


Sample output of sentencepiece

In [14]:
tokenizer = spm.SentencePieceProcessor(model_file=f'{sentencepiece_prefix}.model', add_bos=True, add_eos=True)
sample_tweet = np.random.choice(tweets)
sample_tweet_pieces = tokenizer.encode_as_pieces(sample_tweet)
sample_tweet_encoded = tokenizer.encode(sample_tweet)
print("Sample Tweet:")
print(sample_tweet)
print()
print("Sentence Pieces:")
print(sample_tweet_pieces)
print()
print("Encoded ID's")
print(sample_tweet_encoded)

Sample Tweet:
" @ debdew2: @ brithume @ megynkelly @ DiamondandSilk FOX: TRUMP HATERS R SCARED BECAUSE THEY LOSE - TRUMP IS WINNING pic.twitter.com/6umGLYjWhT"

Sentence Pieces:
['<s>', '▁"', '▁@', '▁de', 'b', 'd', 'ew', '2', ':', '▁@', '▁b', 'ri', 'th', 'um', 'e', '▁@', '▁me', 'g', 'y', 'n', 'k', 'el', 'ly', '▁@', '▁D', 'i', 'am', 'on', 'd', 'and', 'S', 'il', 'k', '▁F', 'O', 'X', ':', '▁TRUMP', '▁H', 'A', 'T', 'ER', 'S', '▁R', '▁S', 'C', 'A', 'RE', 'D', '▁B', 'E', 'C', 'A', 'US', 'E', '▁THE', 'Y', '▁L', 'O', 'S', 'E', '▁-', '▁TRUMP', '▁I', 'S', '▁W', 'I', 'N', 'N', 'ING', '▁pic', '.', 'twitter', '.', 'com', '/', '6', 'um', 'G', 'L', 'Y', 'j', 'W', 'h', 'T', '"', '</s>']

Encoded ID's
[1, 33, 6, 172, 54, 16, 411, 114, 27, 6, 138, 121, 107, 274, 10, 6, 160, 32, 19, 12, 61, 140, 63, 6, 151, 35, 122, 105, 16, 179, 50, 186, 61, 134, 80, 261, 27, 744, 162, 56, 52, 403, 50, 169, 69, 67, 56, 512, 93, 90, 65, 67, 56, 398, 65, 617, 117, 195, 80, 50, 65, 205, 744, 30, 50, 115, 66, 109, 109, 714,

Tokenize tweets. Create input dataset

In [15]:
tokenized_tweets = tokenizer.encode(list(tweets))

class SentencePredictionDataset(torch.utils.data.Dataset):
    """
    Create dataset for sentence predictions
    """
    def __init__(self, tokenized_inputs):
        """
        Initialize dataset from tokenized_inputs
        """
        super(SentencePredictionDataset).__init__()
        self._display_sample(tokenized_inputs)

        # Generate training samples
        samples = []
        for tweet in tqdm(tokenized_tweets, desc='Generating samples'):
            samples.extend(self._create_training_samples_from_sentence(tweet))
        print('Number of:')
        print('\tUnique sequences:', len(tokenized_inputs))
        print('\tTraining samples:', len(samples))

        # Create tensors
        tensors = [
            torch.tensor(sample[0], dtype=torch.int64) 
            for sample in tqdm(samples, desc='Creating input tensors')
        ]
        print('Padding input sequences...')
        self.inputs = torch.nn.utils.rnn.pad_sequence(tensors)
        self.masks = (self.inputs > 0)
        self.labels = torch.tensor([s[1] for s in samples], dtype=torch.int64)

        # Display sample tensor
        print('Results:')
        self._display_size()
        self._display_sample_tensor()

    def _create_training_samples_from_sentence(self, sentence):
        """
        Generate training samples from input sentence (sequence, next-token)
        """
        return [ (sentence[:i], sentence[i]) for i in reversed(range(1, len(sentence))) ]

    def _display_size(self):
        """ Display size """
        print('Size:')
        print('\tInput:', self.inputs.size())
        print('\tMask:', self.masks.size())
        print('\tLabel:', self.labels.size())

    def _display_sample(self, tokenized_inputs):
        """ Display sample of what we're doing """
        print('We\'ll be creating training samples for each tweet like so:')
        sample_seq = choice([ s for s in tokenized_inputs if len(s) <= 10 ])
        print('Encoded Sequence:', sample_seq)
        samples = self._create_training_samples_from_sentence(sample_seq)
        print('Samples:')
        max_seq_len = len(str(samples[0][0]))
        fmt = f'\tSequence: {{0:<{max_seq_len}}}, Label: {{1}}'
        for sequence, label in samples:
            print(fmt.format(str(sequence), label))

    def _display_sample_tensor(self):
        sample_idx = np.random.choice(len(self))
        sequence, mask, label = self[sample_idx]
        print('Sequence:', sequence)
        print('Mask:', mask)
        print('Label:', label)

    def __len__(self):
        """ Length of dataset """
        return self.inputs.size()[-1]

    def __getitem__(self, index):
        """ Get sample """
        return (
            self.inputs[:,index], 
            self.masks[:,index], 
            self.labels[index]
        )

# Create dataset
dataset = SentencePredictionDataset(tokenized_tweets)
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.7, 0.3])
train_ldr = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_ldr = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
print('Datasets created!')

We'll be creating training samples for each tweet like so:
Encoded Sequence: [1, 6, 181, 431, 188, 190, 228, 256, 5, 2]
Samples:
	Sequence: [1, 6, 181, 431, 188, 190, 228, 256, 5], Label: 2
	Sequence: [1, 6, 181, 431, 188, 190, 228, 256]   , Label: 5
	Sequence: [1, 6, 181, 431, 188, 190, 228]        , Label: 256
	Sequence: [1, 6, 181, 431, 188, 190]             , Label: 228
	Sequence: [1, 6, 181, 431, 188]                  , Label: 190
	Sequence: [1, 6, 181, 431]                       , Label: 188
	Sequence: [1, 6, 181]                            , Label: 431
	Sequence: [1, 6]                                 , Label: 181
	Sequence: [1]                                    , Label: 6


Generating samples:   0%|          | 0/43352 [00:00<?, ?it/s]

Number of:
	Unique sequences: 43352
	Training samples: 2303782


Creating input tensors:   0%|          | 0/2303782 [00:00<?, ?it/s]

Padding input sequences...
Results:
Size:
	Input: torch.Size([263, 2303782])
	Mask: torch.Size([263, 2303782])
	Label: torch.Size([2303782])
Sequence: tensor([  1, 668,  14,  90,  17,   3, 422,  60,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  

Create that model!

In [16]:
class TrumpFormerPredictor(torch.nn.Module):
    """ The Donald... but a Robot """
    def __init__(self, d_model=latent_size, vocab_size=vocab_size, n_layers=attention_layers, n_heads=attention_heads):
        super().__init__()
        # Just borrowed from pytorch
        self.vocab_size = vocab_size
        self.embedding = torch.nn.Embedding(self.vocab_size, d_model)
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=n_layers)
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, sequence, mask=None):
        embeddings = self.embedding(sequence)
        transformed = self.transformer(embeddings, mask=mask, is_causal=True)
        logits = transformed[:,-1] @ self.embedding.weight.T
        if self.training:
            return logits
        else:
            return self.softmax(logits)

# Instantiate model, loss, and optimizer
model = TrumpFormerPredictor().to(device)
print(model)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

sequence, mask, label = next(iter(test_ldr))
one_hot = torch.nn.functional.one_hot(label, num_classes=model.vocab_size).float()
print('Sequence:', sequence)
print('Mask:', mask)
print('Label:', label)
print('One-Hot Encoded:', one_hot)

model.train(False)
probs = model(sequence, mask)
predictions = torch.distributions.categorical.Categorical(probs).sample()
print('Output probabilities:', probs)
print('Predictions:', predictions)

TrumpFormerPredictor(
  (embedding): Embedding(1000, 256)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (softmax): Softmax(dim=1)
)
Sequence: tensor([[  1,  92, 293,  ...,   0,   0,   0],
        [  1, 232,  67,  ...,   0,   0,   0],
        [  1,  92, 162,  ...,   0,   0,   0],
        ...,
        [  1, 202,  47,  ...,   0,   0, 

We do a bit of training

In [17]:
def train_model(train_ldr, model, loss_fn, optimizer):
    """
    Process every batch in the training data and
    update model parameters using loss function
    and optimizer (runs one training epoch)
    """
    model.train()
    progress = tqdm(train_ldr, desc='Training:', unit='batch')
    for batch in progress:
        sequences, masks, labels = tuple(b.to(device) for b in batch)
        one_hot_encoded = torch.nn.functional.one_hot(labels, num_classes=model.vocab_size).float()
        logits = model(sequences, masks)
        loss = loss_fn(logits, one_hot_encoded)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        progress.set_postfix({ 'loss': loss.item() })

train_model(train_ldr, model, loss_fn, optimizer)

Training::   0%|          | 0/67194 [00:00<?, ?batch/s]

KeyboardInterrupt: 